<a href="https://colab.research.google.com/github/jaslindoniya/Customer-churn-ML-framing/blob/main/ML_Problemframing_Responsible_Datacard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
from google.colab import files

print("Upload your customer-churn-training csv here")
uploaded = files.upload() #  csv file select

# File name
import io
file_name = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[file_name]))
print(f"Data loaded: {df.shape}")
df.head()

Upload your customer-churn-training csv here


Saving customer-churn-training (1).csv to customer-churn-training (1) (1).csv
Data loaded: (12, 7)


,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0


In [8]:
# EDA
print("Churn count:")
print(df['churned'].value_counts())

print("\nAverage by churn:")
print(df.groupby('churned').mean(numeric_only=True))

# Missing check
print("\nMissing values:")
print(df.isnull().sum())

Churn count:
churned
0    7
1    5
Name: count, dtype: int64

Average by churn:
         tenure_months  support_tickets  monthly_spend_inr  last_login_days
churned                                                                    
0            17.857143         1.142857        1127.571429         3.714286
1             4.200000         3.800000         619.000000        18.800000

Missing values:
customer_id          0
tenure_months        0
support_tickets      0
monthly_spend_inr    0
last_login_days      0
plan_type            0
churned              0
dtype: int64


In [9]:
# Non-ML Baseline
from sklearn.metrics import classification_report, accuracy_score

def rule_baseline(row):
    # which rule for my data tenure minimum + ticket high churn
    if row['tenure_months'] <= 6 and row['support_tickets'] >= 3 and row['last_login_days'] > 10:
        return 1
    return 0

df['baseline_pred'] = df.apply(rule_baseline, axis=1)

print(classification_report(df['churned'], df['baseline_pred']))
print(f"Baseline Accuracy: {accuracy_score(df['churned'], df['baseline_pred'])*100:.1f}%")

# Leakage check - customer_id droped
df_model = df.drop(columns=['customer_id', 'baseline_pred'])
print("\nFeatures for ML:", df_model.columns.tolist())

              precision    recall  f1-score   support

           0       0.88      1.00      0.93         7
           1       1.00      0.80      0.89         5

    accuracy                           0.92        12
   macro avg       0.94      0.90      0.91        12
weighted avg       0.93      0.92      0.91        12

Baseline Accuracy: 91.7%

Features for ML: ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days', 'plan_type', 'churned']


In [10]:
# Risk & Metrics explain
print("""
Business Metrics:
- FP Cost (offer waste) = Rs.500
- FN Cost (customer loss) = Rs.5000
- So FN is 10x costly -> Need high Recall

Abstention Rule:
if model confidence 0.4-0.6 -> send to human

Rollback:
if Precision < 40% for 2 weeks -> go back to rule baseline
""")


Business Metrics:
- FP Cost (offer waste) = Rs.500
- FN Cost (customer loss) = Rs.5000
- So FN is 10x costly -> Need high Recall

Abstention Rule:
if model confidence 0.4-0.6 -> send to human

Rollback:
if Precision < 40% for 2 weeks -> go back to rule baseline

